# Demo · Agente preditivo de performance de YouTube Shorts

Demonstracao executavel (TAP, secao 10): o agente recebe criativos + contexto, **processa a thumbnail** (SigLIP 2) e **o video** (Gemini assiste ao Short pela URL publica e preenche a rubrica do TAP + transcricao), busca Shorts comparaveis do canal e consulta o **modelo final** (CatBoost do E-13). Devolve previsao, intervalo de 80%, confianca, fatores (SHAP), comparaveis e explicacao.

**De-para:** os criativos da demo sao de canais **fora do treino** (FlatOut Brasil, Riachuelo, FARM Rio). As curtidas reais ficam escondidas da tool e so aparecem na comparacao no fim.

**Entrada:** arraste `demo_bundle.tar.gz` (pasta `case_galaxies/.tmp_transfer/`) para o painel Arquivos e cadastre `GEMINI_API_KEY` em *Secrets* (icone de chave, com acesso liberado ao notebook). Com a chave, o Gemini analisa o video de cada criativo e escreve a explicacao. Sem ela, a demo roda com o modelo reserva (so thumbnail) e a explicacao deterministica do SHAP, marcando confianca baixa.

In [ ]:
import os, sys, json, tarfile, subprocess, importlib, warnings
warnings.filterwarnings("ignore")
for pkg, mod in (("catboost", "catboost"), ("langgraph", "langgraph"), ("langchain-google-genai", "langchain_google_genai"), ("mcp", "mcp")):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
WORK = os.environ.get("DEMO_WORK", "/content/demo")
if not os.path.exists(os.path.join(WORK, "src")):
    bundle = next(p for p in ("/content/demo_bundle.tar.gz", os.path.join(WORK, "demo_bundle.tar.gz")) if os.path.exists(p))
    os.makedirs(WORK, exist_ok=True)
    with tarfile.open(bundle) as tar:
        tar.extractall(WORK)
os.environ["GALAXIES_ROOT"] = WORK
sys.path.insert(0, os.path.join(WORK, "src"))
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("Gemini: chave encontrada nos Secrets")
except Exception:
    print("Gemini: sem chave (explicacao deterministica)")
import pandas as pd
from galaxies_agent import Predictor, CreativeInput
from galaxies_agent.agent import build_agent
predictor = Predictor()
agent = build_agent(predictor)
demo = json.load(open(os.path.join(WORK, "demo_creatives.json"), encoding="utf-8"))
print("modelo:", predictor.version, "|", len(demo), "criativos de demo")

## 1. O que o modelo e (inspecao exigida pelo TAP)

In [ ]:
info = predictor.model_info()
print(json.dumps({k: info[k] for k in ("model_version", "reference_experiment", "dataset_version", "training_scope",
                                       "training_rows", "training_channels", "final_test", "target", "rubric",
                                       "baseline_rule")}, indent=1, ensure_ascii=False))
print("modelos:", {k: v["reference"] for k, v in info["models"].items()})

## 2. Caso principal: qual destes 3 Shorts priorizar?

FlatOut Brasil (automotivo, criador, **fora do treino**) publicou 3 Shorts na mesma manha. O agente ranqueia sem ver as curtidas, com os tres avaliados no mesmo instante (antes de qualquer um ser publicado), como numa decisao real.

In [ ]:
def entrada(item):
    return {k: v for k, v in item.items() if not k.startswith("_") and k != "grupo"}

manha = [d for d in demo if d["grupo"] == "FlatOut Brasil" and d["planned_datetime"].startswith("2026-04-18")]
decisao = min(d["planned_datetime"] for d in manha)   # decisao antes de publicar: mesmo historico para os tres
out = agent.invoke({"pedido": "Qual destes 3 Shorts priorizar no FlatOut Brasil?",
                    "criativos": [dict(entrada(d), planned_datetime=decisao) for d in manha]})
print(out["resposta"])
print("\n[fonte da explicacao]", out["explicacao_fonte"], "| numeros verificados:", out["verificacao"]["ok"])

In [ ]:
real = {d["creative_id"]: d["_curtidas_reais"] for d in demo}
tab = pd.DataFrame([{"rank_previsto": r["rank"], "criativo": r["creative_id"], "previsto": r["predicted_metric"],
                     "intervalo_80": f"{r['interval_low']}-{r['interval_high']}", "real": real[r["creative_id"]],
                     "confianca": r["confidence_flag"], "modelo": r["model_variant"], "video": r["rubric_source"],
                     "thumbnail": r["embedding_source"]} for r in out["resultado"]["ranking"]])
tab["rank_real"] = tab["real"].rank(ascending=False).astype(int)
display(tab)
print("Ordem prevista == ordem real?", list(tab["rank_previsto"]) == list(tab["rank_real"]))

### O que o agente viu em cada video (rubrica do Gemini, entra no modelo final)

In [ ]:
campos = ["hook_primeiros_3s", "forca_hook_0a10", "tem_rosto", "pessoa_fala_para_camera", "close_up_produto",
          "texto_na_tela", "tem_musica", "tom_narracao", "chamada_para_acao", "estetica_geral_0a10", "potencial_viral_0a10"]
vistos = {r["creative_id"]: (r["video_attributes"] or {}) for r in out["resultado"]["ranking"]}
display(pd.DataFrame({k: {c: v.get(c) for c in campos} for k, v in vistos.items()}))
for r in out["resultado"]["ranking"]:
    print(r["rank"], r["creative_id"], "| fatores:", [(d["label"], d["value"], d["effect_log"]) for d in r["top_drivers"][:4]])

## 3. De-para completo: previsao x real nos 9 criativos fora do treino

Cada criativo e previsto na propria data de publicacao, com o historico do canal ate ali (mesma regra do Gold). Limitacao conhecida: as curtidas dos posts anteriores sao as da data da coleta, nao as que existiam naquele dia.

In [ ]:
rows = []
for d in demo:
    p = predictor.predict(CreativeInput(**entrada(d)))
    rows.append({"canal": d["grupo"], "criativo": d["creative_id"], "data": d["planned_datetime"][:10],
                 "previsto": p.predicted_metric, "intervalo_80": f"{p.interval_low}-{p.interval_high}",
                 "real": d["_curtidas_reais"], "dentro_do_intervalo": p.interval_low <= d["_curtidas_reais"] <= p.interval_high,
                 "x_mediana_canal_prev": p.relative_to_channel, "confianca": p.confidence_flag, "modelo": p.model_variant,
                 "motivos": "; ".join(p.confidence_reasons)[:120]})
dp = pd.DataFrame(rows)
display(dp)
print(f"Dentro do intervalo de 80%: {dp['dentro_do_intervalo'].mean():.0%} ({dp['dentro_do_intervalo'].sum()}/{len(dp)})")

## 4. Um criativo novo (edite e rode)

Cole a URL de um Short publico e o canal. Para canal da base use `channel_id`; para canal externo use `channel_stats` com as curtidas dos Shorts anteriores.

In [ ]:
novo = {
    "creative": "https://www.youtube.com/shorts/LmWlQe-iAFk",       # troque pela URL do criativo
    "caption": "Titulo planejado do Short",
    "planned_datetime": "2026-10-20T18:00",
    "channel_id": demo[0]["channel_id"],                              # ou remova e use channel_stats
    "sector": "automotivo",
}
resposta = agent.invoke({"pedido": "Quanto esse Short deve render?", "criativos": [novo]})
print(resposta["resposta"])
print("\nmodelo:", resposta["resultado"]["model_variant"], "| video:", resposta["resultado"]["rubric_source"])
print("gancho visto pelo Gemini:", (resposta["resultado"]["video_attributes"] or {}).get("hook_primeiros_3s"))
print("fatores:", [(d["label"], d["effect_log"]) for d in resposta["resultado"]["top_drivers"]])
print("comparaveis:", [(c["title"][:50], c["likes"]) for c in resposta["resultado"]["comparables"]])

## 5. A mesma tool exposta via MCP (como a persona da Galaxies vai consumir)

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def mcp_demo():
    params = StdioServerParameters(command=sys.executable, args=["-m", "galaxies_agent.mcp_server"],
                                   env={**os.environ, "PYTHONPATH": os.path.join(WORK, "src")})
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            tools = await s.list_tools()
            print("tools MCP:", [t.name for t in tools.tools])
            res = await s.call_tool("prever_criativo", {"criativo": dict(entrada(manha[0]), planned_datetime=decisao)})
            body = json.loads(res.content[0].text)
            print({k: body[k] for k in ("predicted_metric", "interval_low", "interval_high", "confidence_flag",
                                        "performance_band", "model_variant", "rubric_source")})

await mcp_demo()